# Análise de Inferência Causal usando DoWhy

Este notebook carrega o conjunto de dados em `data/calculate_data.csv`, importa a definição do DAG causal do arquivo `model/causal_dag.py` e executa a análise de inferência causal passo a passo utilizando a biblioteca **DoWhy**.

In [1]:
import os
import sys
import re
import pandas as pd
import numpy as np
import dowhy
from dowhy import CausalModel

# Garante que o diretório de execução seja a raiz do projeto para localizar os arquivos corretamente
if os.getcwd().endswith('notebooks'):
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# Exibir todas as colunas do dataset
pd.set_option('display.max_columns', None)

/home/vscode/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Carregando os Dados

In [3]:
df = pd.read_csv('data/raw/calculate_data.csv', sep=';')
print(f"Dimensões do dataset: {df.shape}")
df.head()

Dimensões do dataset: (50, 72)


,Property_Name,Location,Owner_Name,Information_Responsible,Predominant_Soil_Type,Lactating_Cows,Dry_Cows,Heifers,Calves,Bulls,Lactating_Cows_Weight,Dry_Cows_Weight,Heifers_Weight,Calves_Weight,Bulls_Weight,Females,Males,Cows_Discarded_Per_Year,Discarded_Cows_Destination,Productivity,Total_Milk_Production,Milk_Protein_Percentage,Milk_Fat_Percentage,Grass_Fresh,Grass_Hay_On_Farm,Grass_Hay_Off_Farm,Grass_Silage_On_Or_Off_Farm,Maize_Silage,Lucerne_Alfalfa_Silage,Other_Fodder,Straw_Peanut_Hull_Others,Crop_Residues,Hay_Oat_Clover,Sorghum_Silage,Wheat,Barley,Maize_Grain,Oats,Rice,Rye,Sorghum,Other_Cereals,DDG,Soybean_Meal_Hull,Sugarbeet_Molasses,Whey_Powder,Rapeseed_Cake_Meal,Sunflower_Cake_Meal,Cottonseed_Cake_Meal,Beans_Peas,Cassava_Tapioca,Other_Feed,Milk_Powder_Dry_Matter,Citrus_Pulp,Mineral_Mix,Lucerne_Alfalfa_Hay,Brewers_Grain_Brewery_Residue,Total,DMI_Check,Diesel,Gasoline,Other_Fuel,Electricity_Grid,Photovoltaic_Energy,Pasture_Area,Non_Organic_Fertilizer,Organic_Fertilizer,co2_enteric_fermentation,co2_manure_management,co2_fertilizer_emissions,co2_energy_emissions,co2_total_emissions
0,BR – 1,Santa Catarina,Anonymous,Anonymous,Latossolo Vermelho-Amarelo,13.0,4.0,8.0,15.0,0.0,500.0,530.0,320.0,150.0,0.0,0.0,0.0,4.0,venda,15.465629,73384.41,3.32,4.13,41241.35,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,47168.95,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,88410.30,88410.30,0.0,0.0,0.0,2500.0,0.0,3.0,2304.0,0.0,51000.0,3471.45,7209.2160,582.5,62263.1660
1,BR – 2,Santa Catarina,Anonymous,Anonymous,Latossolo Vermelho-Amarelo,26.0,5.0,12.0,19.0,0.0,500.0,530.0,320.0,150.0,0.0,0.0,0.0,15.0,venda,17.117211,162442.33,3.19,3.96,0.00,18980.0,0.00,6935.0,131400.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,56684.50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1423.5,0.0,0.0,215423.00,215423.00,2000.0,0.0,0.0,9600.0,0.0,3.5,455.0,0.0,89750.0,6116.70,1423.6950,7596.8,104887.1950
2,BR – 3,Santa Catarina,Anonymous,Anonymous,Latossolo Vermelho-Amarelo,30.0,6.0,20.0,27.0,0.0,500.0,530.0,320.0,150.0,0.0,0.0,0.0,12.0,venda,22.772406,249357.85,2.96,3.35,0.00,2190.0,14924.85,36500.0,100740.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21900.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,115741.50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,291996.35,291996.35,2000.0,0.0,0.0,12000.0,0.0,4.0,854.0,0.0,111000.0,7537.35,2672.1660,8156.0,129365.5160
3,BR – 4,Santa Catarina,Anonymous,Anonymous,Latossolo Vermelho-Amarelo,32.0,10.0,12.0,9.0,0.0,500.0,530.0,320.0,150.0,0.0,0.0,0.0,9.0,venda,16.757430,195726.78,3.48,4.20,89592.90,0.0,0.00,0.0,45990.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2774.0,0.0,0.0,138356.90,138356.90,600.0,0.0,0.0,9600.0,0.0,4.0,696.0,0.0,106000.0,7598.45,2177.7840,3844.8,119621.0340
4,BR – 5,Santa Catarina,Anonymous,Anonymous,Latossolo Vermelho-Amarelo,30.0,4.0,13.0,20.0,0.0,500.0,530.0,320.0,150.0,0.0,0.0,0.0,5.0,venda,19.967890,218648.40,3.31,3.92,153022.60,0.0,0.00,0.0,59130.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,76650.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1642.5,0.0,0.0,290445.10,290445.10,0.0,0.0,0.0,12000.0,0.0,3.0,433.5,0.0,99750.0,6743.20,1356.4215,2796.0,110645.6215


# 2. Importando o grafo causal de model/causal_dag.py

In [4]:
from model.causal_dag import causal_graph

## 3. Inicializando o Modelo Causal (DoWhy)

Instanciamos o `CausalModel`. 
- **Tratamento (Treatment):** `Dietary_Strategy_Cluster` (perfil de estratégia dietética)
- **Desfecho (Outcome):** `co2_enteric_fermentation` (emissões relacionadas à fermentação entérica)
- **Gráfico Causal (Graph):** nosso grafo causal importado acima

As variáveis presentes no DAG mas ausentes no arquivo CSV (como `Dietary_Strategy_Cluster`, `Cattle_Breed`, `Production_System`) serão automaticamente tratadas como variáveis não observadas (confounders latentes) pelo DoWhy.

In [5]:
model = CausalModel(
    data=df,
    treatment='Dietary_Strategy_Cluster',
    outcome='co2_enteric_fermentation',
    graph=causal_graph   
)

/home/vscode/.local/lib/python3.11/site-packages/dowhy/causal_model.py:581: UserWarning: 3 variables are assumed unobserved because they are not in the dataset. Configure the logging level to `logging.WARNING` or higher for additional details.
  warnings.warn(


## 4. Identificando o Efeito Causal (Identificação)

O DoWhy analisa o grafo causal para encontrar caminhos de backdoor/frontdoor e variáveis instrumentais disponíveis para a identificação do efeito.

In [6]:
identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
print(identified_estimand)

Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
             d                                           
───────────────────────────(E[co_2_enteric_fermentation])
d[Dietary_Strategy_Cluster]                              
Estimand assumption 1, Unconfoundedness: If U→{Dietary_Strategy_Cluster} and U→co2_enteric_fermentation then P(co2_enteric_fermentation|Dietary_Strategy_Cluster,,U) = P(co2_enteric_fermentation|Dietary_Strategy_Cluster,)

### Estimand : 2
Estimand name: iv
Estimand expression:
 ⎡                                                                             ↪
 ⎢                            d                                                ↪
E⎢─────────────────────────────────────────────────────────(co_2_enteric_ferme ↪
 ⎣d[Pasture_Area  Production_System  Predominant_Soil_Type]                    ↪

↪                                                                              ↪
↪          ⎛                     